In [1]:
!pip install pandas numpy scikit-learn tensorflow nltk

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

In [3]:
from google.colab import files
uploaded = files.upload()

Saving train.csv to train.csv


In [4]:
import pandas as pd

df = pd.read_csv("train.csv")

df.head()

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


In [5]:
df["cyberbullying"] = df[[
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate"
]].max(axis=1)

In [6]:
df[["comment_text","cyberbullying"]].head()

,comment_text,cyberbullying
0,Explanation\nWhy the edits made under my usern...,0
1,D'aww! He matches this background colour I'm s...,0
2,"Hey man, I'm really not trying to edit war. It...",0
3,"""\nMore\nI can't make any real suggestions on ...",0
4,"You, sir, are my hero. Any chance you remember...",0


In [7]:
df = df[["comment_text","cyberbullying"]]

df.head()

,comment_text,cyberbullying
0,Explanation\nWhy the edits made under my usern...,0
1,D'aww! He matches this background colour I'm s...,0
2,"Hey man, I'm really not trying to edit war. It...",0
3,"""\nMore\nI can't make any real suggestions on ...",0
4,"You, sir, are my hero. Any chance you remember...",0


In [8]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Ensure you have the necessary NLTK data
nltk.download('stopwords')
nltk.download('wordnet')

def clean_text(text):
    # Convert to lowercase
    text = str(text).lower()
    # Remove URLs and HTML tags
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'<.*?>', '', text)
    # Remove special characters and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # Tokenization and removing stopwords/lemmatization
    stop_words = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer()
    words = text.split()
    cleaned_words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words]

    return " ".join(cleaned_words)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


In [9]:
df.shape


(159571, 2)

In [10]:
df["cyberbullying"].value_counts()

,count
cyberbullying,
0,143346
1,16225


In [11]:
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer(num_words=5000)

tokenizer.fit_on_texts(df["comment_text"])

sequences = tokenizer.texts_to_sequences(df["comment_text"])

In [12]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

X = pad_sequences(sequences, maxlen=100)

y = df["cyberbullying"]

In [13]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [14]:
from tensorflow.keras.layers import Bidirectional, SpatialDropout1D

model = Sequential([
    Embedding(input_dim=5000, output_dim=128, input_length=100),
    SpatialDropout1D(0.2), # Prevents overfitting in text data
    Bidirectional(LSTM(64, return_sequences=True)), # Captures context from both directions
    Dropout(0.2),
    LSTM(32),
    Dense(1, activation='sigmoid')
])

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [15]:
history = model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=32,
    validation_data=(X_test, y_test)
)

Epoch 1/5
3990/3990 ━━━━━━━━━━━━━━━━━━━━ 95s 22ms/step - accuracy: 0.9526 - loss: 0.1420 - val_accuracy: 0.9604 - val_loss: 0.1158
Epoch 2/5
3990/3990 ━━━━━━━━━━━━━━━━━━━━ 73s 18ms/step - accuracy: 0.9618 - loss: 0.1069 - val_accuracy: 0.9619 - val_loss: 0.1106
Epoch 3/5
3990/3990 ━━━━━━━━━━━━━━━━━━━━ 71s 18ms/step - accuracy: 0.9650 - loss: 0.0959 - val_accuracy: 0.9602 - val_loss: 0.1117
Epoch 4/5
3990/3990 ━━━━━━━━━━━━━━━━━━━━ 69s 17ms/step - accuracy: 0.9678 - loss: 0.0858 - val_accuracy: 0.9590 - val_loss: 0.1190
Epoch 5/5
3990/3990 ━━━━━━━━━━━━━━━━━━━━ 83s 17ms/step - accuracy: 0.9715 - loss: 0.0767 - val_accuracy: 0.9583 - val_loss: 0.1282


In [16]:
loss, accuracy = model.evaluate(X_test, y_test)

print("Test Accuracy:", accuracy)

998/998 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - accuracy: 0.9583 - loss: 0.1282
Test Accuracy: 0.9582641124725342


In [17]:
predictions = model.predict(X_test)

predictions = (predictions > 0.5).astype(int)

998/998 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step


In [18]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, predictions)

print(cm)

[[28174   497]
 [  835  2409]]


In [19]:
from sklearn.metrics import classification_report

print(classification_report(y_test, predictions))

              precision    recall  f1-score   support

           0       0.97      0.98      0.98     28671
           1       0.83      0.74      0.78      3244

    accuracy                           0.96     31915
   macro avg       0.90      0.86      0.88     31915
weighted avg       0.96      0.96      0.96     31915



In [22]:
model.save("cyberbullying_model.keras")

In [23]:
from tensorflow.keras.models import load_model

model = load_model("cyberbullying_model.keras")

In [25]:
from google.colab import files
files.download("cyberbullying_model.keras")
files.download("tokenizer.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [27]:
from tensorflow.keras.models import load_model
import pickle

model = load_model("cyberbullying_model.keras", compile=False)

with open("tokenizer.pkl", "rb") as f:
    tokenizer = pickle.load(f)

In [26]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
import re

In [28]:
def predict_message(message):

    message = clean_text(message)

    seq = tokenizer.texts_to_sequences([message])

    padded = pad_sequences(seq, maxlen=100)

    prediction = model.predict(padded)[0][0]

    if prediction > 0.5:
        print("Cyberbullying detected")
    else:
        print("Normal message")

    print("Confidence:", prediction)

In [ ]:
predict_message("you are such an idiot")

In [31]:
import io
import json

# 1. Save the Model
model.save("cyberbullying.keras")
print("Model saved as cyberbullying.keras")

# 2. Save the Tokenizer
tokenizer_json = tokenizer.to_json()
with io.open('tokenizer.json', 'w', encoding='utf-8') as f:
    f.write(json.dumps(tokenizer_json, ensure_ascii=False))
print("Tokenizer saved as tokenizer.json")

Tokenizer saved as tokenizer.json


In [32]:
def predict_message(message):
    # Uses the advanced clean_text function from earlier in your notebook
    cleaned = clean_text(message)
    seq = tokenizer.texts_to_sequences([cleaned])
    padded = pad_sequences(seq, maxlen=100)

    prediction = model.predict(padded)[0][0]

    if prediction > 0.5:
        print(f"⚠️ Cyberbullying detected! (Confidence: {prediction*100:.2f}%)")
    else:
        print(f"✅ Normal message (Confidence: {(1-prediction)*100:.2f}%)")

# Test it
predict_message("You are such a wonderful person!")
predict_message("you are such an idiot")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
⚠️ Cyberbullying detected! (Confidence: 53.21%)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step
⚠️ Cyberbullying detected! (Confidence: 99.75%)


In [34]:
from google.colab import files

# Download the exact files needed for your Streamlit App
files.download("cyberbullying_model.keras")
files.download("tokenizer.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>